# Data Layers, Caching & Asynchronous Processing
In this second part of Phase 30, we move beneath the API layer to examine how production systems store data, accelerate read performance with caching, and decouple asynchronous workloads using message brokers.

## 1. Databases: SQL vs. NoSQL
Choosing the right database depends heavily on your data structure, consistency requirements, and scaling needs.

### Relational Databases (SQL - e.g., PostgreSQL, MySQL):

Characteristics: Structured tables with strict schemas, support for ACID transactions (Atomicity, Consistency, Isolation, Durability), and complex relational joins using SQL.

When to use: Financial transactions, user account systems, and highly structured relational data.

### Non-Relational Databases (NoSQL - e.g., MongoDB, Cassandra, DynamoDB):

Characteristics: Flexible schemas (document, key-value, column-family, or graph stores), optimized for horizontal scaling, high write throughput, and unstructured or semi-structured data.

When to use: Real-time analytics, chat applications, large-scale catalogs, and rapid, schema-evolving prototypes.

## 2. Caching with Redis
Redis (Remote Dictionary Server) is an in-memory data structure store used primarily as a blazing-fast cache, session store, or message broker.

**Why use it?** Reading from disk or querying a database takes milliseconds; fetching from Redis RAM takes microseconds. It prevents database overload during traffic spikes.

**Common Use Cases:** Caching expensive API responses, rate-limiting user requests, and managing distributed locks.

In [ ]:
import redis

# Connect to Redis
client = redis.Redis(host='localhost', port=6379, db=0)

# Set a key with an expiration time (TTL) of 60 seconds
client.setex("user:1001:profile", 60, '{"name": "Alice", "role": "Admin"}')

# Retrieve cached data
profile = client.get("user:1001:profile")
if profile:
    print("Cache hit:", profile.decode('utf-8'))
else:
    print("Cache miss: Query database")

## 3. Message Queues: RabbitMQ & Kafka
When background tasks (like sending emails, processing video encoding, or generating reports) take too long to run inside a synchronous HTTP request, you offload them to a message queue.

### RabbitMQ (Traditional Task Queue / AMQP):

Focuses on reliable message delivery, complex routing logic, and task distribution across worker nodes.

When to use: Microservice communication, background job processing (via Celery), and transactional messaging.

### Apache Kafka (Distributed Event Streaming Platform):

Designed for high-throughput, fault-tolerant, append-only commit logs. Messages are retained even after consumption, allowing multiple downstream consumers to replay streams.

When to use: Real-time event-driven architectures, clickstream tracking, log aggregation, and massive big data pipelines.

### Best Practices & Common Pitfalls
**Implement Cache Invalidation Strategies:** Caching is notoriously difficult due to stale data. Always set appropriate TTLs (Time-To-Live) and use cache-aside or write-through patterns to keep data fresh.

**Design for Idempotency in Queues:** Network glitches can cause message queues to deliver the same task twice. Ensure your background workers can process duplicate messages safely without causing double-charges or duplicate records.